# 2 — Steering the retriever: filters, quotas (A) and link expansion (B)

This notebook drives `HierarchicalPGRetriever` directly — no UI, no agent, no LLM — and
shows how each steering mechanism changes the retrieved set. Reference:
[`docs/RETRIEVAL.md`](../RETRIEVAL.md) §7.

**Prerequisites:**
- Neo4j up with the built graph (`scripts/start_services.sh`); the *full* graph lives on
  the GPU host — on a dev machine with the small verify subset you will see tiny,
  homogeneous results.
- `:Document.category` backfilled — notebook
  [01_source_categories.ipynb](01_source_categories.ipynb).
- The embedding model (`BAAI/bge-large-en-v1.5`, ~1.3 GB) downloads on first use; query
  embedding runs fine on CPU.

In [ ]:
# Environment: credentials live in ~/.myenvs/ema_nlp.env (never in the repo).
# On this host the project Neo4j container runs on alt ports; if your env file
# does not set NEO4J_URI, the setdefault below points at the project container.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / ".myenvs" / "ema_nlp.env")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7688")  # project container (alt port)

# Make `harness` importable when the notebook is started from docs/examples/.
REPO = Path.cwd()
while not (REPO / "harness").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("repo root:", REPO)

## 1. Open the index and build the baseline retriever

`open_index` opens the existing Neo4j graph (no rebuild, no re-embed) and configures the
profile's embedder; `build_retriever` dispatches on the profile's retrieval strategy.
This is exactly what the chat app does at startup.

In [ ]:
from harness.indexing import build_retriever, load_index_profile, open_index

profile = load_index_profile("neo4j_hier")   # the plain, steering-OFF profile
index = open_index(profile)
retriever = build_retriever(profile, index)
print("k =", profile.retrieval.k, "| quotas:", profile.retrieval.category_quota,
      "| expansion:", profile.retrieval.graph.expand)

## 2. Baseline: what does an unsteered top-k look like?

A small display helper shows each node's score, category, and retrieval origin — the
same provenance the agent sees as `category=` / `via=` tags in its tool output.

In [ ]:
from collections import Counter


def show(nodes):
    for i, n in enumerate(nodes, 1):
        m = n.node.metadata
        origin = m.get("retrieval_origin", "vector")
        print(f"[{i:2d}] {n.score:.3f}  {m['category']:22s} {origin:15s} "
              f"{(m.get('title') or '')[:55]}")
        print(f"      {m['source_url']}")
    print("\ncategory mix:", dict(Counter(n.node.metadata["category"] for n in nodes)))


QUESTION = "What is the acceptable intake for N-nitrosodimethylamine (NDMA) impurities?"

baseline = retriever.retrieve(QUESTION)
show(baseline)

On the full graph the mix above is typically EPAR-heavy: the corpus contains far more
product assessments than guidelines, and a reorder-only postprocessor could not fix that
— if no guideline made the top-k, there is nothing to float up. The mechanisms below act
on the **candidate set** instead.

## 3. Option A(i) — per-call category filter

`with_categories([...])` returns a *view* of the retriever restricted to the given
categories. Under the hood the vector query **oversamples** (`k × retrieval.oversample`,
default 4) and filters on the persisted `:Document.category` in Cypher, so the final
top-k is drawn from a pool the filter didn't starve. This is the same seam the agent's
`ema_search(source_category=...)` argument uses (notebook 03).

In [ ]:
guidelines_only = retriever.with_categories(["scientific_guideline", "qa"])
show(guidelines_only.retrieve(QUESTION))

## 4. Option A(ii) — category quotas (profile-driven)

A hard filter answers "only show me guidelines"; a **quota** answers "always include
some guidelines". The `neo4j_steered` profile — same graph, same embeddings, nothing
rebuilt — turns on `category_quota` (guaranteed slots within the final k, stratified
from the oversampled pool) **and** link expansion (next section). Quotas are guarantees,
not requirements: a category with no pool members yields its slots back, and the final
list keeps score order — stratification changes *membership*, never ranking.

In [ ]:
steered_profile = load_index_profile("neo4j_steered")
print("quotas:", steered_profile.retrieval.category_quota)
print("oversample:", steered_profile.retrieval.oversample)
print("expansion:", steered_profile.retrieval.graph.expand,
      "->", steered_profile.retrieval.graph.expand_categories,
      "max", steered_profile.retrieval.graph.max_expand)

steered = build_retriever(steered_profile, index)   # same opened index
steered_nodes = steered.retrieve(QUESTION)
show(steered_nodes)

## 5. Option B — `LINKS_TO` expansion

The graph's 99,520 typed `LINKS_TO` edges encode "this page/report cites that document"
— exactly the path from an EPAR hit to the guideline behind it. With
`retrieval.graph.expand: true` the retriever follows those edges from the vector-hit
documents and **appends** the best-matching chunk of up to `max_expand` linked documents
(optionally restricted to `expand_categories` and the edge's
`link_contexts`/`document_types` properties).

Expansion is strictly **additive**: expanded nodes never displace a vector hit, and they
carry full provenance — `retrieval_origin="link_expansion"` plus `linked_from` (the seed
document ids they were reached from). Their scores are cosine similarities rescaled to
the same `[0, 1]` range as the vector-index scores.

In [ ]:
expanded = [n for n in steered_nodes
            if n.node.metadata.get("retrieval_origin") == "link_expansion"]
print(f"{len(expanded)} node(s) arrived via link expansion:\n")
for n in expanded:
    m = n.node.metadata
    print(f"  {m['category']:22s} {m['source_url']}")
    print(f"    linked_from: {m['linked_from']}")

## 6. All knobs, programmatically (no profile file)

Profiles are the normal configuration surface, but every knob is a plain constructor
argument — handy for experiments before committing values to a YAML. The embedder was
configured by `open_index`, so `Settings.embed_model` is the profile's model.

In [ ]:
from llama_index.core import Settings

from harness.indexing.profiles import GraphRetrievalConfig
from harness.indexing.property_graph import HierarchicalPGRetriever

custom = HierarchicalPGRetriever(
    index.property_graph_store,
    Settings.embed_model,
    k=8,
    merge=True,                                   # small-to-big: return parent chunks
    oversample=6,                                 # pool = 48 candidates when steering
    category_quota={"scientific_guideline": 3},   # >= 3 guideline slots (if available)
    graph=GraphRetrievalConfig(
        expand=True,
        expand_categories=["scientific_guideline", "qa"],
        max_expand=2,
    ),
)
show(custom.retrieve(QUESTION))

## Takeaways

- **Filter** = "only these categories" (per call, oversample-and-filter in Cypher).
- **Quota** = "always some of these categories" (per profile, stratified membership,
  score order preserved).
- **Expansion** = "also bring what the hits *cite*" (additive, provenance-tagged).
- Everything is off by default (`neo4j_hier` unchanged); `neo4j_steered` is the
  turn-key steered profile.

Next: **[03_routing_and_full_agent.ipynb](03_routing_and_full_agent.ipynb)** — the
query→category routing table (Option C), the `ema_search` tool standalone, and the full
headless agent pipeline.